<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12_power_grid/Ex12.2_gnn_contingency_screening.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.2 — Screening Contingencies

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L12.2 · Prediction of Power Grid Stability**

When a short circuit occurs on a transmission line, the protection opens the
line after a fixed time, here 140 ms. While the short circuit lasts, the
generators near it cannot deliver their power and their rotors speed up. If
it lasts too long, a generator loses synchronism with the rest of the grid
and has to be disconnected, which is a serious disturbance. The longest short
circuit a generator withstands is its **critical clearing time**. Grid
operators check it for every line that could fail and for every operating
condition: this is called **contingency screening**. Each check needs a
simulation in time, which makes screening slow. You compute the critical
clearing time in two ways - by simulation, and by a graph neural network
trained on simulated results - compare the two on accuracy and computing
time, and then test the network on a line outage it has never seen.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the reference simulation, and a cheaper simulation with a longer time step | — |
| **5** | the same critical case in ANDES, a dedicated power system simulator | — |
| **6** | a graph neural network for the critical clearing time, and the comparison | the graph layer, the loss |
| **7** | a line outage the network has not seen, and the threshold of the screen | — |
| **8** | what the notebook says | — |
| **9 – 10** | your report, and the mini project proposal | your answers |

About two minutes on a CPU, half of it ANDES. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem_12_2.py', 'andes_compare.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12_power_grid/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem_12_2"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem_12_2.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem_12_2 as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex12.2_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

The grid is the one of Ex_12.1: six buses connected by six lines. Bus 0 is the
connection to the Nordic transmission system, bus 1 a local power plant,
buses 2 to 4 three load areas and bus 5 an HVDC connection. Two machines are
modelled: the Nordic system at bus 0, represented as one very large machine
(inertia constant $H$ = 150 s), and the generator of the local plant at bus 1
($H$ = 4 s).

A **contingency** is one possible failure: here a three-phase short circuit at
one end of a line, cleared by the protection opening that line. There are six:
a short circuit at the plant's bus that is cleared without losing a line, and
a short circuit on each of five lines. (The sixth line is not screened: opening
it disconnects bus 5 from the grid, which is a loss of supply, not a stability
question.)

For every operating point and every contingency the task is to find the
**critical clearing time** (CCT): the longest duration of the short circuit
after which the plant still stays in synchronism with the rest of the grid.
**A case is insecure if its critical clearing time is 140 ms or less**: the
protection would open the line too late.

## 2 · The data

| | |
|---|---|
| operating points | 180, varying the total load (0.85 to 1.60 of the base load), the share of it produced by the local plant, and the HVDC import |
| contingencies | 6 per operating point, so 1080 cases |
| what is known for each bus | 6 numbers before the short circuit: the injected active and reactive power $P$ and $Q$, the voltage magnitude $V$ and angle $\theta$, whether a machine is connected there, whether the short circuit occurs there |
| what is known about the lines | which buses are connected after the line is opened: a 6 × 6 table of ones and zeros per case (the adjacency matrix of the grid) |
| what is wanted | the critical clearing time of each case, searched up to 500 ms |
| training and test | 144 operating points to train on, 36 kept apart to test - always whole operating points, never single cases |

**The grid is representative of eastern Denmark, not a model of it**, and the
plant's inertia, damping and reactance are estimates for a unit of its size.
The machines are modelled in the simplest way that shows the effect (the
classical model: no voltage regulator, no turbine governor, first swing only).

## 3 · The physics

**The swing equation** (L12.2's *Transient Stability*). The rotor angle
$\delta_i$ of each machine changes with its speed $\omega_i$, and the speed
changes with the difference between the mechanical power driving the rotor
and the electrical power it delivers to the grid:

$$\frac{d\delta_i}{dt} = \omega_i - \omega_s, \qquad
  \frac{2H_i}{\omega_s}\frac{d\omega_i}{dt} = P_{m,i} - P_{e,i}(\delta) - D_i\,\frac{\omega_i - \omega_s}{\omega_s}$$

with $H_i$ the inertia constant, $D_i$ the damping and $\omega_s$ the
synchronous speed. During the short circuit the plant can deliver almost no
electrical power, so its rotor speeds up and its angle grows. When the line is
opened it can deliver power again - over a grid with one line fewer - and
slows down, provided the angle has not grown too far.

**The critical clearing time** is the longest short circuit after which the
angle between the two machines stays below 180°; beyond 180° the plant loses
synchronism.

The cell below draws the grid with one contingency, prints the
problem, and shows one case cleared 10 ms before its critical clearing time
and 10 ms after it.

In [ ]:
pb.draw_grid(); plt.show()                                   # the grid, one line out and its fault
pb.describe_problem()
cases = pb.contingencies()                                    # the six contingencies
base = np.stack(pb.injections())                              # the base operating point: P and Q at every bus
cct_one, one = pb.label_case(base, cases[1]["outage"], cases[1]["fault_bus"], return_detail=True)
print(f"  base operating point, line 0 out: critical clearing time {1000 * cct_one:.0f} ms")

fig, ax = plt.subplots(figsize=(8, 3.2))
for extra, style in ((-0.010, "-"), (0.010, "--")):
    T, D, W = pb.simulate_swing(one["machines"], one["Yred_pre"], one["Yred_post"], t_end=1.5, t_fault=pb.T_FAULT,
                                t_clear=pb.T_FAULT + cct_one + extra, Yred_fault=one["Yred_fault"], delta0=one["delta0"])
    ax.plot(T, np.degrees(D[:, 1] - D[:, 0]), style, lw=2, label=f"cleared after {1000 * (cct_one + extra):.0f} ms")
ax.axhline(180, color="0.5", lw=1); ax.set_ylim(0, 360)
ax.set_xlabel("time  [s]"); ax.set_ylabel("angle between the machines  [deg]"); ax.legend(frameon=False); plt.tight_layout(); plt.show()

**What you should see.** The six contingencies, and for line 0 out at the
base operating point a critical clearing time of 220 ms: the case is secure,
with 80 ms to spare. Cleared 10 ms before that time, the angle between the
machines rises past 120°, turns and comes back. Cleared 10 ms after it, the
angle keeps growing: the plant has lost synchronism. **Twenty milliseconds
separate a harmless event from the loss of the plant.**

---

## 4 · The reference, and a cheaper simulation

**Why a simulation in time.** The swing equations are nonlinear differential
equations: the electrical power depends on the sine of the angle. For a grid
like this one they have no solution in closed form, and the critical
clearing time has no formula. (The equal area criterion of L12.2 gives it in
closed form only for one machine against an infinitely large grid, without
damping.) The equations are therefore **integrated step by step in time**,
with the fourth-order Runge–Kutta method (RK4): the time is divided into
steps of 2 ms, and from the state at one step the method computes the state
at the next. Unlike Newton–Raphson in Ex_12.1, this is a discretisation: its
accuracy depends on the size of the time step.

**Finding the critical clearing time.** One simulation answers only whether a
given clearing time is safe. The critical clearing time is found by
**bisection**: try the middle of an interval known to contain it, simulate,
keep the half that still contains it, and repeat until the interval is 2 ms
wide. That takes about eight simulations per case. `pb.build_cases` does this
for all 1080 cases at once.

**Four methods with four roles.**

| | role here |
|---|---|
| RK4 with a 2 ms step, bisection to 2 ms | the **reference**: the critical clearing times all others are scored against |
| RK4 with a longer step (5, 10, 20 ms) | the **classical way to save time**: fewer steps, less accuracy |
| ANDES (section 5) | a **check**: the same case in a dedicated power system simulator |
| a graph neural network (section 6) | the **learned estimator**, trained on the reference |

**The agreed accuracy is 5 ms** on the worst case: under 4 % of the
protection's 140 ms. The cell below builds the reference, then repeats it
with longer steps.

In [ ]:
data = pb.build_cases(180, seed=12)                           # the reference: step 2 ms, bisection to 2 ms
X, A, cct, op_id, cont_id = data["X"], data["A"], data["cct"], data["op_id"], data["cont_id"]
t_ref = float(data["seconds"]) / len(cct)
insecure = cct <= pb.PROTECTION_TIME
print(f"  reference: {len(cct)} cases, {1000 * t_ref:.2f} ms a case; {insecure.sum()} insecure ({100 * insecure.mean():.0f} %), "
      f"{(cct >= pb.CCT_MAX).sum()} at the 500 ms ceiling")
for c in cases:
    m = cont_id == c["index"]
    print(f"    {c['label'][:44]:44s} median {1000 * np.median(cct[m]):3.0f} ms, insecure in {100 * insecure[m].mean():2.0f} %")

AGREED = 0.005
coarse = {}
for step, tol in ((0.005, 0.004), (0.010, 0.008), (0.020, 0.016)):
    d = pb.build_cases(180, seed=12, dt=step, tol=tol)        # the same simulation, coarser
    err = np.abs(d["cct"] - cct)
    wrong = int(((d["cct"] <= pb.PROTECTION_TIME) != insecure).sum())
    coarse[step] = (float(err.mean()), float(err.max()), wrong, float(d["seconds"]) / len(cct), d["cct"])
    print(f"  step {1000 * step:2.0f} ms: mean error {1000 * err.mean():.2f} ms, worst {1000 * err.max():.2f} ms, "
          f"{wrong} verdicts changed, {1000 * coarse[step][3]:.2f} ms a case" + ("   <- meets 5 ms" if err.max() <= AGREED else ""))
STEP = 0.005                                                  # the coarsest step that meets it

**What you should see.** 1080 cases at a few milliseconds each; 16 % of them
insecure - with line 0, 1 or 3 out or the short circuit at the plant, none
with line 2 or 4 out - and 17 % at the 500 ms limit of the search. **A step of
5 ms is the longest that meets the agreed 5 ms** (worst error 4.9 ms), at
about a third of the cost. Even so it changes 19 verdicts: many cases lie
within a few milliseconds of 140 ms, and for those a small error in the
clearing time changes the answer.

On a real grid one case does not take a millisecond: there are thousands of
buses and detailed machine models, and a screening covers every line for
every hour of the day.

---

## 5 · The same case in ANDES

The simulation of section 4 is a few dozen lines of Python written for this
course. A stability study in practice is made with a dedicated simulator.
**ANDES** is one: open source, written in Python, it builds the grid from its
own library of models, solves the power flow, and integrates the machine
equations with an implicit method (Cui, Li & Tomsovic, *IEEE Trans. Power
Systems*, 2021).

The cell installs it on Colab (about 30 s) and finds the critical clearing
time of the most critical contingency - line 0 out, the short circuit at the
plant, the base operating point - three ways, each by the same bisection:

* the course's simulation (RK4, 2 ms step, the classical machine model);
* ANDES with the same classical machine model (GENCLS);
* ANDES with a detailed machine model (GENROU) with its voltage regulator
  (ESST1A) and turbine governor (TGOV1), with typical parameters - what a
  study in practice would use.

In [ ]:
import contextlib, io, subprocess, sys
try:
    import andes
except ImportError:                                           # on Colab: install ANDES once, about 30 s
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "andes"], check=True)
import andes_compare as ac
import andes, logging
andes.config_logger(stream_level=50); logging.getLogger("andes").setLevel(logging.CRITICAL)   # keep ANDES quiet

c = ac.contingencies()[1]                                     # the worst contingency: line 0 out, the fault at the plant
op = np.stack(pb.injections())                                # the base operating point
solvers = (("course simulation, classical machine", lambda: pb.label_case(op, c["outage"], c["fault_bus"])),
           ("ANDES, classical machine", lambda: ac.compute_cct(c["fault_bus"], c["outage"], model="classical")),
           ("ANDES, machine with exciter and governor", lambda: ac.compute_cct(c["fault_bus"], c["outage"], model="detailed")))
andes_rows = []
for name, find in solvers:
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):   # ANDES reports every run it stops
        cct_x = find()
    andes_rows.append((name, cct_x, time.perf_counter() - t0))
print(f"  {c['label']}, base operating point")
print(f"  {'':44s}{'critical clearing time':>24s}{'margin to 140 ms':>18s}{'time to find it':>17s}")
for name, cct_x, sec in andes_rows:
    print(f"  {name:44s}{1000 * cct_x:20.1f} ms{1000 * (cct_x - pb.PROTECTION_TIME):15.1f} ms{sec:15.1f} s")

**What you should see.** About 220 ms from the course's simulation, 211 ms
from ANDES with the same classical machine model, and 166 ms with the
detailed one. **The same model in two simulators agrees within about 9 ms**:
ANDES applies the short circuit at the bus itself, the course's simulation as
a very small impedance to earth. **The detailed machine model gives about
53 ms less**, leaving only about 27 ms of margin to the protection's 140 ms:
the classical model this notebook's network learns from is optimistic, and a
network can be no better than the values it is trained on. ANDES needed 10 to
20 s for one critical clearing time against under a second for the course's
simulation - for all 1080 cases of section 4, four to six hours. That is the
cost a learned screen is meant to save.

---

## 6 · A graph neural network for the critical clearing time

**What the network does.** For one case it takes the numbers known before
the short circuit at every bus and the grid that remains after the line is
opened, and returns the critical clearing time. In general terms: a grid of
$B$ buses, $F$ numbers per bus, one critical clearing time out. Here $B$ = 6
and $F$ = 6 (the table of section 2).

**It is not a PINN.** The network is trained by supervised learning: it is
shown the critical clearing times of the reference simulation and learns to
reproduce them. No equation appears in its loss. The physics is in the
values it is trained on, and in the graph it is given (L12.2's *Three Places
to Put the Physics*).

**Why a graph neural network.** A contingency changes the grid: a line is
removed. A dense network takes a fixed list of numbers and would have to be
told which line is out by an extra code; it has no notion of which buses are
connected. A graph neural network is given the grid itself, as the adjacency
matrix, and computes with it: every bus combines its own numbers with those of
the buses it is connected to. Its weights are the same for every bus, so the
same network works for every contingency - and can be asked about a grid with
a line out that it has never seen (section 7).

**One graph layer** computes, for every bus at once,

$$H' = \tanh\!\big(H\,W_{\mathrm{self}} + b + \hat{A}\,H\,W_{\mathrm{neigh}}\big)$$

with $H$ the numbers at the $B$ buses (one row per bus), $W_{\mathrm{self}}$
and $W_{\mathrm{neigh}}$ two weight matrices shared by all buses, $b$ a bias,
and $\hat{A}$ the adjacency matrix with each bus also connected to itself,
scaled by the number of connections of each bus. One layer passes
information one line further. This grid is three lines across, so there are
three layers; then the mean over the buses, and a small dense network that
returns one number.

**TODO 1 is the graph layer** - one line: a bus's own numbers through one
weight matrix, its neighbours' sum through another.

In [ ]:
class GraphLayer(torch.nn.Module):                            # a network: layers in __init__, the computation in forward
    def __init__(self, n_in, n_out):
        super().__init__()
        self.own = torch.nn.Linear(n_in, n_out)               # dense layer on a bus's own numbers, with the bias
        self.neighbours = torch.nn.Linear(n_in, n_out, bias=False)   # dense layer on what the neighbours send
    def forward(self, A_hat, H):
        # TODO 1: the graph layer of section 6, before its tanh. A_hat has shape (N, 6, 6), H has shape (N, 6, features).
        #
        #   return self.own(H) + A_hat @ self.neighbours(H)       # own numbers, plus the neighbours' summed along the lines
        raise NotImplementedError("Write the graph layer")

**TODO 2 is the loss** - one line, and no equation in it: the mean squared
difference between the network's and the reference critical clearing time.
The physics is in the values it learns from, not in the loss.

In [ ]:
def loss_fn(model, A_t, H_t, y_t):
    # TODO 2: the loss - the mean squared distance between the network's output and the labels y_t.
    #
    #   return (model(A_t, H_t) - y_t).pow(2).mean()              # mean squared distance from the simulated clearing time
    raise NotImplementedError("Write the loss")

**The network and its training**

| | |
|---|---|
| type | graph neural network (graph convolution), followed by a dense network |
| input | per case: $F$ = 6 numbers at each of the $B$ = 6 buses, each scaled to zero mean and unit standard deviation over the training cases, and the adjacency matrix after the line is opened |
| graph layers | 3, of 32 neurons each, activation $\tanh$; from the second layer on, each layer's output is added to its input (a residual connection) |
| pooling | the mean over the buses: one vector of 32 numbers per case |
| dense network | 2 hidden layers of 32 neurons, activation $\tanh$, one output: the critical clearing time in units of 100 ms |
| weights | 6 721, Xavier initialisation, random seed 88 |
| training data | the 864 cases of 144 operating points, with their reference critical clearing times |
| loss | the mean squared difference between the network's and the reference critical clearing time |
| optimiser | Adam, 1500 steps, learning rate $3\times10^{-3}$, then L-BFGS, 40 steps - the Part 2 recipe |
| batch | all training cases at every step |
| test data | the 216 cases of the 36 operating points kept apart |

**Training proceeds in three steps, repeated:** the network returns a
critical clearing time for every training case; it is compared with the
reference value from the simulation; the weights are adjusted to make the
difference smaller.

In [ ]:
class Screen(torch.nn.Module):                                # a network: layers in __init__, the computation in forward
    def __init__(self, hidden=32, depth=3):
        super().__init__()
        self.layers = torch.nn.ModuleList([GraphLayer(pb.N_FEATURE if k == 0 else hidden, hidden) for k in range(depth)])   # three graph layers
        self.head = MLP(n_in=hidden, n_out=1, n_hidden=32, n_layers=2)   # dense tanh network: the pooled numbers in, one number out
    def forward(self, A_hat, H):
        for k, layer in enumerate(self.layers):
            Z = torch.tanh(layer(A_hat, H))                   # one graph layer, then tanh
            H = H + Z if k > 0 else Z                         # a residual connection after the first layer
        return self.head(H.mean(dim=-2))                      # the mean over the six buses, then the dense head

A_hat = pb.normalised_adjacency(A)                            # self-loops added, scaled by the degrees
train = op_id < 144                                           # 144 operating points to train on, 36 held out

def fit(keep):
    # a network trained on the cases `keep`; returns it and a function from case indices to predicted CCT in seconds
    set_seed(88)                                              # fix the random start so a run repeats
    flat = X[keep].reshape(-1, pb.N_FEATURE)
    mean, spread = flat.mean(axis=0), flat.std(axis=0)
    Xs = (X - mean) / np.where(spread > 1e-12, spread, 1.0)   # every feature to zero mean and unit spread, on the training cases
    # the graphs and the bus numbers of the cases m, as tensors on the device
    tensors = lambda m: (to_tensor(A_hat[m]).reshape(-1, 6, 6).to(DEVICE), to_tensor(Xs[m]).reshape(-1, 6, pb.N_FEATURE).to(DEVICE))   # NumPy arrays -> tensors
    model = Screen().to(DEVICE)                               # the network, on the device
    A_t, H_t = tensors(keep)                                  # the training graphs and bus numbers
    y_t = to_tensor(cct[keep] / 0.1).to(DEVICE)               # the labels, in units of 100 ms
    t0 = time.perf_counter()
    # Adam first, then L-BFGS: the Part 2 training recipe
    train_two_stage(model, lambda: loss_fn(model, A_t, H_t, y_t), adam_steps=1500, lbfgs_steps=40, lr=3e-3, report_every=0)
    seconds = time.perf_counter() - t0
    def predict(m):
        with torch.no_grad():  # no gradient tracking: evaluation only
            return to_numpy(model(*tensors(m))).ravel() * 0.1  # back to seconds
    return model, predict, seconds

Training is the Part 2 recipe - Adam, then L-BFGS - with the loss of TODO 2,
all training cases at every step. About half a minute.

In [ ]:
net, predict, train_time = fit(train)
print(f"{parameter_count(net)} weights, {train.sum()} training cases, trained in {train_time:.0f} s")

The comparison: the network against the 5 ms simulation on the 216 test
cases - the error of the critical clearing time, the insecure cases missed,
the false alarms, and the time per case.

In [ ]:
test = ~train                                                 # the 36 held-out operating points: 216 cases
pred = predict(test)
t_net = middle(lambda: predict(test)) / test.sum()            # all 216 at once

def row(name, p, truth, seconds, training="-"):
    c = pb.confusion(truth, p)                                # the verdicts at 140 ms: missed insecure cases, false alarms
    err = np.abs(p - truth)
    rms = np.sqrt((err ** 2).mean())
    print(f"  {name:34s}{1000 * err.mean():8.2f} ms{1000 * rms:8.2f} ms{1000 * err.max():8.2f} ms{c['missed']:6d} of {c['tp'] + c['fn']:2d}{c['alarms']:9d}{1e6 * seconds:10.0f} us{training:>10s}")
    return err.mean(), err.max(), c["missed"], c["alarms"], rms

print(f"{'216 test cases':36s}{'mean error':>11s}{'RMSE':>11s}{'worst':>11s}{'missed':>12s}{'alarms':>9s}{'one case':>13s}{'training':>10s}")
print(f"  {'reference, step 2 ms':34s}{'-':>11s}{'-':>11s}{'-':>11s}{'-':>12s}{'-':>9s}{1e6 * t_ref:10.0f} us{'-':>10s}")
r_coarse = row("simulation, step 5 ms", coarse[STEP][4][test], cct[test], coarse[STEP][3])
r_net = row("graph neural network", pred, cct[test], t_net, f"{train_time:.0f} s")

e_coarse = 1000 * np.abs(coarse[STEP][4][test] - cct[test])   # the errors of the 5 ms simulation, in ms
e_net = 1000 * np.abs(pred - cct[test])                        # the errors of the network, in ms
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
ax[0].boxplot([e_coarse, e_net], showfliers=True, flierprops=dict(markersize=3))
ax[0].set_xticks([1, 2], ["simulation, 5 ms step", "graph neural network"])
ax[0].axhline(1000 * AGREED, color="0.4", ls="--", lw=1); ax[0].text(0.55, 1000 * AGREED, "agreed 5 ms", va="bottom", fontsize=8)
ax[0].set_ylabel("error of the critical clearing time  [ms]")
ax[0].set_title("error on the 216 test cases: lower is better, 0 is perfect", fontsize=9)
cid = cont_id[test]; k = np.arange(len(cases)); wdt = 0.38
ax[1].bar(k - wdt / 2, [e_coarse[cid == j].mean() for j in k], wdt, label="simulation, 5 ms step")
ax[1].bar(k + wdt / 2, [e_net[cid == j].mean() for j in k], wdt, label="graph neural network")
ax[1].set_xticks(k, ["plant" if c["outage"] is None else f"line {c['outage']}" for c in cases])
ax[1].set_xlabel("contingency: where the short circuit is cleared"); ax[1].set_ylabel("mean error  [ms]")
ax[1].set_title("mean error for each contingency", fontsize=9); ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(cc.output_path("Ex12.2_report.png"), dpi=150); plt.show()

**How to read the results.** For each method the table gives the **mean
error** and the **root-mean-square error (RMSE)** of the critical clearing
time over the 216 test cases, the **worst** error, and the two counts that
matter for a screen: **missed** insecure cases (the screen says secure, the
simulation says insecure) and **false alarms** (the other way round). Ideally
all would be zero. A missed case is the dangerous error - a short circuit the
protection cannot clear in time - while a false alarm only costs one more
simulation.

**What you should see.** On operating points it has never seen, the network
is within about 0.6 ms on average and 5 ms at worst: **more exact than the
simulation with the 5 ms step**, with no insecure case missed and no false
alarm (the 5 ms simulation raises 5), at a few microseconds a case against
about a millisecond. Training took about half a minute, and the 864 values
it was trained on took the reference a few seconds.

**The figure.** The left panel shows the spread of the errors of the two
methods (the box holds the middle half of the cases, the line in it is the
median, the dots are single cases far from the rest); the dashed line is the
agreed 5 ms. The right panel shows the mean error for each contingency. The
5 ms simulation has an error of a few milliseconds on every contingency,
because its error comes from the time step; the network's error is smaller
everywhere.

Read that with care. The network does not solve the swing equation. It has
learned how the critical clearing time varies with load and import **on six
grids it has seen hundreds of times each**.

---

## 7 · A line outage the network has not seen, and the threshold of the screen

**Why this test.** In operation the grid changes: lines are taken out for
maintenance, and a screen must answer for grids it was not trained on. The
fair test (L12.2's *How to Find Out Whether the Physics Helped*) is therefore
a grid that was never in the training set. The cell trains the same network
again, on the same 144 operating points, **without any case that has line 0
out**, and then asks it about line 0 out on the 36 test operating points.
Line 0 is the direct connection between the plant and the Nordic system, and
its loss is the most critical of the six contingencies.

**The threshold of the screen.** A screen sends a case to the full simulation
if the predicted critical clearing time is below a threshold. At 140 ms it
sends what it believes insecure. Raising the threshold makes it miss fewer
insecure cases and raise more false alarms (L12.2's *Screening, Not
Replacement*). The cell raises it until no insecure case is missed.

In [ ]:
line0 = cont_id == 1                                          # the cases with line 0 out
net_u, predict_u, _ = fit(train & ~line0)                     # trained without ever seeing line 0 out
held = test & line0                                           # line 0 out, on the test operating points
pred_seen, pred_u = predict(held), predict_u(held)
print(f"{'line 0 out, 36 test cases':36s}{'mean error':>11s}{'worst':>11s}{'missed':>12s}{'alarms':>9s}")
for name, p in (("network that has seen line 0 out", pred_seen), ("network that has not", pred_u)):
    c = pb.confusion(cct[held], p); err = np.abs(p - cct[held])
    print(f"  {name:34s}{1000 * err.mean():8.2f} ms{1000 * err.max():8.2f} ms{c['missed']:6d} of {c['tp'] + c['fn']:2d}{c['alarms']:9d}")
bias = float(np.mean(pred_u - cct[held]))
print(f"  the network that has not seen it is {1000 * bias:+.0f} ms off on average: it says the grid is safer than it is")

thresholds = np.arange(0.140, 0.3001, 0.005)
bad = cct[held] <= pb.PROTECTION_TIME                         # the truly insecure cases
missed = [int((bad & ~(pred_u <= th)).sum()) for th in thresholds]
alarms = [int((~bad & (pred_u <= th)).sum()) for th in thresholds]
first = int(np.argmax(np.array(missed) == 0))                 # the lowest threshold that misses nothing
print(f"  threshold raised from 140 ms to {1000 * thresholds[first]:.0f} ms: no insecure case missed, {alarms[first]} false alarms among {int((~bad).sum())} secure cases")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.0))
ax[0].axvspan(0, 140, ymin=0, ymax=1, color="tab:red", alpha=0.08)
ax[0].plot([0, 520], [0, 520], color="0.5", ls="--", lw=1, label="perfect prediction")
ax[0].plot(1000 * cct[held], 1000 * pred_seen, "o", ms=4, label="network that has seen line 0 out")
ax[0].plot(1000 * cct[held], 1000 * pred_u, "s", ms=4, label="network that has not")
ax[0].axvline(140, color="tab:red", lw=1); ax[0].axhline(140, color="tab:red", lw=1)
ax[0].text(5, 405, "truly insecure\n(simulated CCT under 140 ms)", fontsize=8, va="top")
ax[0].set_xlim(0, 420); ax[0].set_ylim(0, 420)
ax[0].set_xlabel("simulated critical clearing time  [ms]"); ax[0].set_ylabel("predicted  [ms]")
ax[0].set_title("line 0 out: a point above its red line is a missed insecure case", fontsize=9); ax[0].legend(frameon=False, fontsize=8, loc="lower right")
ax[1].plot(1000 * thresholds, missed, "o-", ms=3, label="insecure cases missed")
ax[1].plot(1000 * thresholds, alarms, "s-", ms=3, label="false alarms")
ax[1].axvline(1000 * thresholds[first], color="0.5", ls="--", lw=1); ax[1].text(1000 * thresholds[first] + 2, 14, f"{1000 * thresholds[first]:.0f} ms: nothing missed", fontsize=8)
ax[1].set_xlabel("threshold of the screen  [ms]"); ax[1].set_ylabel("number of the 36 cases")
ax[1].set_title("the network that has not seen line 0 out: raising the threshold", fontsize=9); ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

**What you should see.** In the left panel the network that has seen line 0
out lies on the diagonal. **The network that has not lies about 56 ms above
it, always on the same side**: it predicts the clearing time of a grid that
still has its strongest line, and so **it misses nearly all of the 11
insecure cases** (10 in our run; the count can move by one with the device
and the library) - the squares above the red horizontal line, left of the red
vertical one.

In the right panel, raising the threshold from 140 ms to 185 ms removes every
missed case and raises no false alarm. The network is off by an almost
constant amount, so it still puts the cases in the right order. **But 185 ms
was found with the true answers known.** For a line that has never been
simulated, nobody knows how far to raise it.

---

## 8 · What the notebook says

* **The simulation gives the true answer here, and it is what costs time.**
  About eight simulations of the swing equations per case, for every line and
  every operating point. A longer time step is the classical saving, and at
  5 ms it already changes verdicts near the limit.
* **On grids it has seen, the network was faster and more exact than the
  cheaper simulation.** That is interpolation over load and import, on six
  grids, and for that it is very good.
* **On a grid it had not seen, it was wrong, and wrong on the unsafe side.**
  A graph neural network can be asked about a new grid, which a dense
  network cannot. Being able to ask is not the same as getting the right
  answer.
* **It still ranked the unseen cases correctly.** Its error was a nearly
  constant offset, so a raised threshold separated them. The right threshold
  was only known because the true answers were.
* **So it screens; it does not decide.** The cases it is unsure of, and every
  grid outside its training, go to the simulation.
* **There is no physics in this network's loss.** The physics is in the
  values it learns from and in the graph. L12.2's physics-informed graph
  network, with the swing equation in the loss, is not built here; whether it
  does better on the unseen line is the experiment the lecture describes.
* **A margin says more than a verdict.** The clearing time minus 140 ms tells
  an operator how close a case is. Cases a few milliseconds from the limit
  are the ones a small error changes.
* **The values are the classical machine model's**: no voltage regulator, no
  turbine governor, first swing only, on a grid whose impedances are textbook
  values. ANDES with a detailed machine model put the most critical case
  about 57 ms closer to the limit (section 5); a network trained on the
  classical values inherits their optimism.

---

## 9 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex12.2_report.md` with the results of section 6 and 7 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L12.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Why is screening combinatorial, and what did one case cost the reference in this notebook? Say what that cost becomes on a grid of thousands of buses, as far as the notebook lets you. (-> L12.2 Q1)": """
""",
    "Section 3 showed one fault cleared 10 ms before its critical time and 10 ms after. Explain the two curves with the equal area criterion, and say why the clearing time minus 140 ms is worth more to an operator than the verdict. (-> L12.2 Q1)": """
""",
    "A line switched out is an edge removed. What did the graph network do with that, in your own TODO 1, that a dense network on a list of 36 numbers could not? (-> L12.2 Q3)": """
""",
    "Compare the network on held-out operating points with the network on a held-out topology. Which of the two tests would you report, what did it take to make the screen miss nothing, and could you have known that threshold without the simulation? (-> L12.2 Q4)": """
""",
    "To conclude: would you put this network in front of an operator, and for what? Say what it may decide, what it must pass on to the simulation, and what you would do before trusting it on a line it has not seen. (-> L12.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| 216 test cases | mean error | RMSE | worst | insecure missed | false alarms | one case | training |", "|---|---|---|---|---|---|---|---|",
            f"| reference, step 2 ms | - | - | - | - | - | {1e6 * t_ref:.0f} us | - |",
            f"| simulation, step 5 ms | {1000 * r_coarse[0]:.2f} ms | {1000 * r_coarse[4]:.2f} ms | {1000 * r_coarse[1]:.2f} ms | {r_coarse[2]} | {r_coarse[3]} | {1e6 * coarse[STEP][3]:.0f} us | - |",
            f"| graph neural network | {1000 * r_net[0]:.2f} ms | {1000 * r_net[4]:.2f} ms | {1000 * r_net[1]:.2f} ms | {r_net[2]} | {r_net[3]} | {1e6 * t_net:.0f} us | {train_time:.0f} s |",
            "", f"Line 0 out, never seen in training: {1000 * np.abs(pred_u - cct[held]).mean():.2f} ms off on average, bias {1000 * bias:+.0f} ms, "
            f"{pb.confusion(cct[held], pred_u)['missed']} of {int(bad.sum())} insecure cases missed",
            "", f"Threshold for no missed case: {1000 * thresholds[first]:.0f} ms, with {alarms[first]} false alarms among {int((~bad).sum())} secure cases",
            "", "The grid's impedances are textbook values; the labels are the classical machine model's."]
    out = ["# Ex_12.2 — Screening Contingencies", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex12.2_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 6, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex12.2_report.md into Ex12.2_report.pdf, with any figure
# saved as Ex12.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex12.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex12.2_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex12.2_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex12.2_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 10 · Mini projects

Each mini project takes this set's problem further - a harder version of it, or its
inverse - with no exact solution. Each student chooses **one** mini project from the
Part 2 sets and solves it individually in one month; the dates and the hand-in are
given in L13. Each mini project is **one Colab notebook in five parts** under course
policy C14: A the problem, B the data and the physics and C the reference model and
the ground truth are given and solved; D the physics-informed network and E the
report are yours. The goal is the reference model's accuracy with an online answer
at least ten times faster (for an inverse problem, ten times faster than a new
least-squares fit); each project's README states the problem, what changed, the
ground truth, a worked example and the requirements.

| mini project | new against the exercise | required | notebook |
|---|---|---|---|
| [**MP12A · Stability Estimation after Contingencies**](https://github.com/RemusTeodorescu/DL-for-Engineers-Public-Course/tree/main/miniprojects/MP12A) | the IEEE 39-bus grid, 35 contingencies, 0 to 60 % of the generation from converters with no inertia | critical clearing time within 5 ms on average, no insecure case missed, at most 5 % false alarms | [open in Colab](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP12A/MP12A_stability_estimation.ipynb) |
| [**MP12B · Stability Prediction after a Contingency**](https://github.com/RemusTeodorescu/DL-for-Engineers-Public-Course/tree/main/miniprojects/MP12B) | after the fault: 200 ms of noisy PMU-like records predict the rotor angles to 3 s and the verdict | at most 1 % of losses of synchronism missed and 5 % false alarms, angles within 5° (median) | [open in Colab](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP12B/MP12B_stability_prediction.ipynb) |